# VisualCOT-Swap — Full Pilot (All 3 Models, Single Session)

**Runs:** LLaVA-1.5-7B → InternVL2-8B → Idefics2-8B, 20 samples each  
**Expected time:** ~90–110 min on Colab A100  
**Output:** `results/pilot/{model}/pilot_summary.json` per model + combined plot  

**Optimizations for limited GPU hours:**
- Attention rollout attribution (no backprop — 3× faster than Grad-CAM, works on 4-bit)
- `max_new_tokens=300` for pilot speed
- Model unloaded + `empty_cache()` between models
- All results auto-saved to Google Drive after each model
- Resumes from Drive checkpoint if session disconnects

---
**Before running:** Push `~/Insta 3/` code to `https://github.com/jeeth-kataria/CoherenceTax.git`  
Or: manually upload the `src/` and `scripts/` folders to `/content/` and skip Cell 3.

In [ ]:
# ── CELL 1: Install ───────────────────────────────────────────────────────────
import subprocess, sys

pkgs = [
    'transformers==4.47.0', 'accelerate==0.27.0', 'bitsandbytes==0.42.0',
    'datasets==2.20.0', 'Pillow', 'scipy', 'scikit-learn',
    'matplotlib', 'seaborn', 'opencv-python-headless', 'spacy', 'tqdm'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)
print('✓ Install done')

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/visualcot'
os.makedirs(f'{DRIVE_ROOT}/data/scienceqa', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data/swap_pairs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results/pilot', exist_ok=True)
print(f'✓ Drive mounted → {DRIVE_ROOT}')

In [ ]:
# ── CELL 3: Clone repo ────────────────────────────────────────────────────────
import os, subprocess

REPO_URL = 'https://github.com/jeeth-kataria/CoherenceTax.git'
REPO_DIR = '/content/visualcot'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✓ Repo at {REPO_DIR}')

In [ ]:
# ── CELL 4: Download ScienceQA (cached on Drive) ──────────────────────────────
import shutil, json

LOCAL_DATA = f'{REPO_DIR}/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

sqa_meta = f'{LOCAL_DATA}/scienceqa/metadata.json'
drive_sqa_meta = f'{DRIVE_ROOT}/data/scienceqa/metadata.json'

if os.path.exists(drive_sqa_meta):
    print('Restoring ScienceQA from Drive...')
    shutil.copytree(f'{DRIVE_ROOT}/data/scienceqa', f'{LOCAL_DATA}/scienceqa', dirs_exist_ok=True)
    with open(sqa_meta) as f:
        meta = json.load(f)
    print(f'✓ ScienceQA loaded from Drive: {len(meta)} records')
else:
    print('Downloading ScienceQA (first time, ~5 min)...')
    result = subprocess.run(
        [sys.executable, 'scripts/prepare_data.py',
         '--output', LOCAL_DATA, '--max_scienceqa', '8000', '--max_gqa', '0'],
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    # Cache to Drive
    shutil.copytree(f'{LOCAL_DATA}/scienceqa', f'{DRIVE_ROOT}/data/scienceqa', dirs_exist_ok=True)
    print('✓ ScienceQA downloaded and cached to Drive')

In [ ]:
# ── CELL 5: Build 20-triplet benchmark (cached on Drive) ─────────────────────
import json, shutil

LOCAL_BM  = f'{LOCAL_DATA}/swap_pairs/benchmark_metadata.json'
DRIVE_BM  = f'{DRIVE_ROOT}/data/swap_pairs/benchmark_metadata.json'

if os.path.exists(DRIVE_BM):
    print('Restoring benchmark from Drive...')
    os.makedirs(f'{LOCAL_DATA}/swap_pairs', exist_ok=True)
    shutil.copy(DRIVE_BM, LOCAL_BM)
    with open(LOCAL_BM) as f:
        bm = json.load(f)
    print(f'✓ Benchmark loaded from Drive: {len(bm)} triplets')
else:
    print('Building 20-triplet benchmark...')
    result = subprocess.run(
        [sys.executable, 'scripts/build_benchmark.py',
         '--n', '20', '--output', f'{LOCAL_DATA}/swap_pairs', '--dry_run'],
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    shutil.copy(LOCAL_BM, DRIVE_BM)
    print('✓ Benchmark built and cached to Drive')

In [ ]:
# ── CELL 6: Shared imports + helpers ─────────────────────────────────────────
import gc, json, time, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from datasets import load_dataset
from tqdm.notebook import tqdm

from src.models.loader import load_model_and_processor, build_prompt, parse_steps, unload_model, load_config
from src.attribution.rollout import attention_rollout, find_image_token_range
from src.metrics.gfs import compute_gfs_sequence, compute_decay_slope, summarize_result
from src.utils.stats import save_checkpoint, load_checkpoint

CFG = load_config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = CFG['attribution']['image_size']  # 336
MAX_STEPS = CFG['gfs']['max_steps']          # 6
MAX_NEW_TOKENS = 300  # faster for pilot
N_SAMPLES = 20

print(f'✓ Device: {DEVICE}')
print(f'✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'✓ Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Load ScienceQA HF dataset (image access)
print('Loading ScienceQA HF dataset...')
SQA_DS = load_dataset('derek-thomas/ScienceQA', split='train', trust_remote_code=True)

# Load benchmark
with open(LOCAL_BM) as f:
    TRIPLETS = json.load(f)
print(f'✓ Benchmark: {len(TRIPLETS)} triplets')

def get_image(hf_index):
    img = SQA_DS[hf_index].get('image')
    if img and isinstance(img, Image.Image):
        return img.convert('RGB')
    return None

def run_one_sample(model, processor, model_cfg, model_key, triplet):
    orig = triplet['original']
    swap_meta = triplet.get('swap')
    question = orig['question']

    orig_img = get_image(orig['hf_index'])
    if orig_img is None:
        return None
    swap_img = get_image(swap_meta['hf_index']) if swap_meta else None

    n_patches  = model_cfg.get('n_image_patches', 576)
    tok_id     = model_cfg.get('image_token_id', -200)

    def get_gfs(img):
        prompt = build_prompt(model_key, processor, question, cot=True)
        inputs = processor(text=prompt, images=img, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        gen_ids = out[0][inputs['input_ids'].shape[1]:]
        text = processor.decode(gen_ids, skip_special_tokens=True)
        steps = parse_steps(text)[:MAX_STEPS]

        img_start, img_end = find_image_token_range(inputs['input_ids'], tok_id, n_patches)
        heatmaps = []
        for s in steps:
            step_inputs = processor(
                text=build_prompt(model_key, processor, s, cot=False),
                images=img, return_tensors='pt'
            ).to(DEVICE)
            hm = attention_rollout(model, step_inputs, img_start, img_end, IMG_SIZE)
            heatmaps.append(hm)
        return text, steps, compute_gfs_sequence(heatmaps, steps)

    cot_text, steps, gfs_orig = get_gfs(orig_img)
    _, swap_steps, gfs_swap = get_gfs(swap_img) if swap_img else (None, [], None)

    result = {
        'triplet_id': triplet['triplet_id'],
        'model_key':  model_key,
        'question':   question,
        'subject':    orig.get('subject', ''),
        'cot_text':   cot_text,
        'steps':      steps,
        'gfs_per_step': gfs_orig,
        'gfs_per_step_swapped': gfs_swap,
        'n_steps': len(steps),
    }
    result.update(summarize_result(result))
    return result

print('✓ Helpers ready')

In [ ]:
# ── CELL 7: Run function (call once per model) ────────────────────────────────
def run_model_pilot(model_key):
    out_dir = Path(f'{REPO_DIR}/results/pilot/{model_key}')
    drive_dir = Path(f'{DRIVE_ROOT}/results/pilot/{model_key}')
    out_dir.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = str(out_dir / 'checkpoint.json')
    drive_ckpt = str(drive_dir / 'checkpoint.json')

    # Resume from Drive checkpoint
    completed = []
    if os.path.exists(drive_ckpt):
        shutil.copy(drive_ckpt, ckpt_path)
        completed = load_checkpoint(ckpt_path)
        print(f'Resumed: {len(completed)} done')

    done_ids = {r['triplet_id'] for r in completed}
    todo = [t for t in TRIPLETS if t['triplet_id'] not in done_ids][:N_SAMPLES]

    if not todo:
        print(f'{model_key}: already complete')
        return completed

    print(f'\n{"="*50}')
    print(f'Running {model_key} — {len(todo)} samples')
    print(f'Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')
    print(f'{"="*50}')

    model, processor, model_cfg = load_model_and_processor(model_key)
    results = list(completed)
    t0 = time.time()
    errors = 0

    for i, triplet in enumerate(tqdm(todo, desc=model_key)):
        try:
            res = run_one_sample(model, processor, model_cfg, model_key, triplet)
            if res is None:
                continue
            results.append(res)
            # Save per-sample JSON
            with open(out_dir / f"{res['triplet_id']}.json", 'w') as f:
                json.dump(res, f, default=str)
        except torch.cuda.OutOfMemoryError:
            print(f'OOM on {triplet["triplet_id"]} — skipping')
            torch.cuda.empty_cache()
            errors += 1
            continue
        except Exception as e:
            print(f'Error: {e}')
            errors += 1
            continue

        # Checkpoint every 5 samples
        if (i + 1) % 5 == 0:
            save_checkpoint(results, ckpt_path)
            shutil.copy(ckpt_path, drive_ckpt)
            elapsed = time.time() - t0
            eta = (elapsed / (i+1)) * (len(todo) - i - 1)
            print(f'  [{i+1}/{len(todo)}] ETA {eta/60:.1f} min')

    save_checkpoint(results, ckpt_path)
    shutil.copy(ckpt_path, drive_ckpt)

    # Pilot summary
    valid_gfs    = [r['gfs_mean'] for r in results if r.get('gfs_mean') is not None]
    valid_steps  = [r['n_steps'] for r in results]
    valid_slopes = [r['gfs_decay_slope'] for r in results if r.get('gfs_decay_slope') is not None]

    summary = {
        'model':             model_key,
        'n_samples':         len(results),
        'n_errors':          errors,
        'avg_gfs':           float(np.mean(valid_gfs))  if valid_gfs    else None,
        'avg_steps':         float(np.mean(valid_steps)) if valid_steps  else None,
        'avg_decay_slope':   float(np.mean(valid_slopes)) if valid_slopes else None,
        'samples_ge3_steps': sum(1 for s in valid_steps if s >= 3),
        'elapsed_min':       round((time.time() - t0) / 60, 1),
    }

    summary_path = out_dir / 'pilot_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    shutil.copy(summary_path, drive_dir / 'pilot_summary.json')

    # Unload model
    unload_model(model)
    gc.collect()

    print(f'\n✓ {model_key} done: {json.dumps(summary, indent=2)}')
    return results

print('✓ run_model_pilot() ready')

In [ ]:
# ── CELL 8: Run LLaVA-1.5-7B ─────────────────────────────────────────────────
results_llava = run_model_pilot('llava')

In [ ]:
# ── CELL 9: Run InternVL2-8B ──────────────────────────────────────────────────
results_internvl = run_model_pilot('internvl')

In [ ]:
# ── CELL 10: Run Idefics2-8B ──────────────────────────────────────────────────
results_idefics2 = run_model_pilot('idefics2')

In [ ]:
# ── CELL 11: Results + GFS decay plot ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

MODEL_COLORS = {'llava': '#534AB7', 'internvl': '#1D9E75', 'idefics2': '#D85A30'}
MODEL_LABELS = {'llava': 'LLaVA-1.5-7B', 'internvl': 'InternVL2-8B', 'idefics2': 'Idefics2-8B'}

all_results = {
    'llava':    results_llava,
    'internvl': results_internvl,
    'idefics2': results_idefics2,
}

# ── Print summaries ───────────────────────────────────────────────────────────
print('='*60)
print('PILOT SUMMARY')
print('='*60)
for mk, results in all_results.items():
    summary_path = Path(f'{REPO_DIR}/results/pilot/{mk}/pilot_summary.json')
    if summary_path.exists():
        with open(summary_path) as f:
            s = json.load(f)
        print(f'\n{MODEL_LABELS[mk]}')
        print(f'  samples        : {s["n_samples"]}')
        print(f'  avg_steps      : {s["avg_steps"]:.2f}')
        print(f'  avg_gfs        : {s["avg_gfs"]:.3f}' if s['avg_gfs'] else '  avg_gfs: N/A')
        print(f'  avg_decay_slope: {s["avg_decay_slope"]:.4f}' if s['avg_decay_slope'] else '  slope: N/A')
        print(f'  steps>=3       : {s["samples_ge3_steps"]}/{s["n_samples"]}')
        wk1 = '✓ WEEK 1 MET' if (s.get('avg_steps') or 0) >= 3 else '✗ avg_steps < 3'
        print(f'  Week 1 check   : {wk1}')

# ── GFS decay plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: GFS decay curve
ax = axes[0]
for mk, results in all_results.items():
    if not results:
        continue
    by_step = [[] for _ in range(MAX_STEPS)]
    for r in results:
        for k, g in enumerate((r.get('gfs_per_step') or [])[:MAX_STEPS]):
            if g is not None and not np.isnan(g):
                by_step[k].append(g)
    means = [np.mean(s) if s else np.nan for s in by_step]
    sems  = [np.std(s)/np.sqrt(len(s)) if len(s)>1 else 0 for s in by_step]
    valid = [i for i, m in enumerate(means) if not np.isnan(m)]
    if not valid:
        continue
    x = np.array(valid) + 1
    y = np.array([means[i] for i in valid])
    e = np.array([sems[i]  for i in valid])
    ax.plot(x, y, 'o-', color=MODEL_COLORS[mk], label=MODEL_LABELS[mk], markersize=5)
    ax.fill_between(x, y-e, y+e, color=MODEL_COLORS[mk], alpha=0.15)

ax.set_xlabel('Reasoning step k'); ax.set_ylabel('Mean GFS')
ax.set_ylim(0, 1); ax.legend(frameon=False)
ax.set_title('GFS Decay — Pilot (n=20 per model)')
ax.spines[['top','right']].set_visible(False)

# Right: per-model mean GFS bar
ax2 = axes[1]
mk_list = [mk for mk in ['llava','internvl','idefics2'] if all_results.get(mk)]
gfs_means = []
gfs_errs  = []
for mk in mk_list:
    vals = [r['gfs_mean'] for r in all_results[mk] if r.get('gfs_mean') is not None]
    gfs_means.append(np.mean(vals) if vals else 0)
    gfs_errs.append(np.std(vals)/np.sqrt(len(vals)) if len(vals)>1 else 0)

bars = ax2.bar(range(len(mk_list)), gfs_means, yerr=gfs_errs,
               color=[MODEL_COLORS[mk] for mk in mk_list],
               capsize=4, alpha=0.8)
ax2.set_xticks(range(len(mk_list)))
ax2.set_xticklabels([MODEL_LABELS[mk] for mk in mk_list], rotation=10, ha='right')
ax2.set_ylabel('Mean GFS (across all steps)')
ax2.set_ylim(0, 1)
ax2.set_title('Mean GFS by Model')
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
fig_path = f'{REPO_DIR}/results/pilot/pilot_combined.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
shutil.copy(fig_path, f'{DRIVE_ROOT}/results/pilot/pilot_combined.png')
plt.show()
print(f'✓ Plot saved')

In [ ]:
# ── CELL 12: Qualitative check — sample CoT output ───────────────────────────
print('='*60)
print('QUALITATIVE SAMPLE (LLaVA, first result)')
print('='*60)

if results_llava:
    r = results_llava[0]
    print(f'Question : {r["question"]}')
    print(f'Subject  : {r["subject"]}')
    print(f'N steps  : {r["n_steps"]}')
    print(f'GFS/step : {[round(g,3) if g is not None else None for g in r["gfs_per_step"]]}')
    print(f'Decay    : {r.get("gfs_decay_slope")}')
    print(f'Swap sens: {r.get("swap_sensitivity_score")}')
    print(f'\nCoT output (first 600 chars):')
    print(r['cot_text'][:600])

print()
print('DIAGNOSIS:')
for mk in ['llava', 'internvl', 'idefics2']:
    results = all_results.get(mk, [])
    if not results:
        print(f'  {mk}: NO DATA')
        continue
    all_gfs = [g for r in results for g in (r.get('gfs_per_step') or []) if g is not None]
    if not all_gfs:
        print(f'  {mk}: All GFS = None → Rollout hooks may not have fired. Check output_attentions support.')
    elif max(all_gfs) - min(all_gfs) < 0.05:
        print(f'  {mk}: GFS nearly constant ({min(all_gfs):.3f}–{max(all_gfs):.3f}) → heatmaps may be uniform')
    else:
        print(f'  {mk}: GFS varies ({min(all_gfs):.3f}–{max(all_gfs):.3f}) ✓')

In [ ]:
# ── CELL 13: Final sync to Drive ─────────────────────────────────────────────
# Always run this before closing. Saves everything.
shutil.copytree(
    f'{REPO_DIR}/results',
    f'{DRIVE_ROOT}/results',
    dirs_exist_ok=True
)
print(f'✓ All results synced to Drive')
print(f'Drive path: {DRIVE_ROOT}/results/')

# Print GPU time used
total_mins = sum(
    json.load(open(f'{REPO_DIR}/results/pilot/{mk}/pilot_summary.json')).get('elapsed_min', 0)
    for mk in ['llava','internvl','idefics2']
    if Path(f'{REPO_DIR}/results/pilot/{mk}/pilot_summary.json').exists()
)
print(f'Total GPU time used: ~{total_mins:.0f} min')